In [ ]:
import anndata as ad
import pandas as pd
from scipy import sparse
import numpy as np
import gc
from pathlib import Path

In [ ]:
path = "../../data/full_data.csv"
reader = pd.read_csv(path, index_col=0, chunksize=100)

blocks, obs_names, var_names = [], [], None
for i,chunk in enumerate(reader):
    if Path(f"../../data/chunk_{i}.h5ad").exists():
        continue
    if var_names is None:
        var_names = chunk.columns.to_numpy()
    obs_names = chunk.index.to_numpy()
    X = sparse.csr_matrix(chunk.to_numpy(dtype=np.float32)).T.tocsr()
    adata = ad.AnnData(
        X=X,
        var=pd.DataFrame(index=obs_names),
        obs=pd.DataFrame(index=var_names),
    )
    adata.write_h5ad(f"../../data/chunk_{i}.h5ad", compression="gzip")
    gc.collect()

In [ ]:
from pathlib import Path
ad.experimental.concat_on_disk([p.name for p in Path("../../data/").glob("chunk_*.h5ad")], "../../data/full_data.h5ad")

In [2]:
anndata.read_csv("../../data/RNA_expression.csv").write_h5ad("../../data/full_data.h5ad")

/home/ntbiotech/.miniforge/envs/scml/lib/python3.11/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


KeyboardInterrupt: 